# PINN v3 — Dead Reckoning + Learned Drift Correction

### Why v2 failed fundamentally
v2 asked an MLP to map `(t, ax, ay, wz)` → `(px, py, vx, vy, yaw)`.  
But a single IMU reading at time `t` tells you **nothing** about absolute position —  
position is the double-integral of acceleration. The network has no memory of the past.

### New approach
1. **Dead-reckon** the IMU: numerically integrate `(ax, ay, wz)` → `(px, py, vx, vy, yaw)_DR`
2. This drifts due to IMU noise and bias
3. **PINN learns the correction**: `network(t)` → `δ = [δpx, δpy, δvx, δvy, δψ]`
4. **Corrected state** = dead_reckoning + δ
5. **Data loss**: `MSE(DR + δ, ground_truth)`
6. **Physics loss**: the corrected state must satisfy the 2D INS equations

The network only needs to learn a **smooth, low-frequency drift** — much easier than  
recovering absolute state from scratch.

In [ ]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## 1. Load & align data (same as v2)

In [ ]:
base = "/kaggle/input/datasets/bhavaniprasadm3/euroc-mav/MH_01_easy"
imu_df = pd.read_csv(f"{base}/mav0/imu0/data.csv")
gt_df  = pd.read_csv(f"{base}/mav0/state_groundtruth_estimate0/data.csv")

gt_times  = gt_df["#timestamp"].values
imu_times = imu_df["#timestamp [ns]"].values
indices   = np.clip(np.searchsorted(imu_times, gt_times, side="left"), 0, len(imu_times) - 1)
imu_aligned = imu_df.iloc[indices].reset_index(drop=True)

# Time in seconds from 0
t = ((gt_times - gt_times[0]) / 1e9).astype(np.float64)
dt = np.diff(t, prepend=t[0])
dt[0] = dt[1]  # fix first dt

print(f"N = {len(t)}, duration = {t[-1]:.1f}s, mean dt = {dt.mean()*1000:.1f}ms")

## 2. Gravity compensation (same as v2)

In [ ]:
ax_raw = imu_aligned["a_RS_S_x [m s^-2]"].values.astype(np.float64)
ay_raw = imu_aligned["a_RS_S_y [m s^-2]"].values.astype(np.float64)
az_raw = imu_aligned["a_RS_S_z [m s^-2]"].values.astype(np.float64)
wz_raw = imu_aligned["w_RS_S_z [rad s^-1]"].values.astype(np.float64)

quats = gt_df[[" q_RS_x []", " q_RS_y []", " q_RS_z []", " q_RS_w []"]].values
rotations = Rotation.from_quat(quats)

g_world  = np.array([0.0, 0.0, 9.81])
g_sensor = rotations.inv().apply(g_world)

ax = (ax_raw - g_sensor[:, 0])
ay = (ay_raw - g_sensor[:, 1])
wz = wz_raw

print(f"After gravity comp — ax mean: {ax.mean():.4f}, ay mean: {ay.mean():.4f}")

In [ ]:
# Ground truth
px_gt  = gt_df[" p_RS_R_x [m]"].values.astype(np.float64)
py_gt  = gt_df[" p_RS_R_y [m]"].values.astype(np.float64)
vx_gt  = gt_df[" v_RS_R_x [m s^-1]"].values.astype(np.float64)
vy_gt  = gt_df[" v_RS_R_y [m s^-1]"].values.astype(np.float64)
yaw_gt = rotations.as_euler("ZYX")[:, 0].astype(np.float64)

## 3. Dead Reckoning — numerically integrate the IMU

This is the key new step. We integrate the gravity-corrected IMU to get a baseline trajectory.  
We initialize from ground truth at `t=0` (in practice you'd use GPS or another source).  
The DR trajectory will **drift** over time due to accumulated noise — that's what the PINN corrects.

In [ ]:
N = len(t)

# Pre-allocate dead-reckoning arrays
psi_dr = np.zeros(N)
vx_dr  = np.zeros(N)
vy_dr  = np.zeros(N)
px_dr  = np.zeros(N)
py_dr  = np.zeros(N)

# Initialize from ground truth at t=0
psi_dr[0] = yaw_gt[0]
vx_dr[0]  = vx_gt[0]
vy_dr[0]  = vy_gt[0]
px_dr[0]  = px_gt[0]
py_dr[0]  = py_gt[0]

# Forward Euler integration
for i in range(1, N):
    d = dt[i]
    # Yaw: integrate gyro
    psi_dr[i] = psi_dr[i-1] + wz[i-1] * d
    # Body→world acceleration
    c, s = np.cos(psi_dr[i-1]), np.sin(psi_dr[i-1])
    ax_w = ax[i-1] * c - ay[i-1] * s
    ay_w = ax[i-1] * s + ay[i-1] * c
    # Velocity: integrate acceleration
    vx_dr[i] = vx_dr[i-1] + ax_w * d
    vy_dr[i] = vy_dr[i-1] + ay_w * d
    # Position: integrate velocity
    px_dr[i] = px_dr[i-1] + vx_dr[i-1] * d
    py_dr[i] = py_dr[i-1] + vy_dr[i-1] * d

# How much does dead reckoning drift?
dr_pos_err = np.sqrt((px_dr - px_gt)**2 + (py_dr - py_gt)**2)
print(f"Dead reckoning position error:")
print(f"  Mean: {dr_pos_err.mean():.3f} m")
print(f"  Max:  {dr_pos_err.max():.3f} m")
print(f"  Final: {dr_pos_err[-1]:.3f} m")

In [ ]:
# Visualize DR vs ground truth
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(px_gt, py_gt, 'k-', label='Ground Truth', linewidth=2)
axes[0].plot(px_dr, py_dr, 'r--', label='Dead Reckoning', linewidth=1)
axes[0].set_xlabel('X (m)'); axes[0].set_ylabel('Y (m)')
axes[0].set_title('2D Trajectory'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

axes[1].plot(t, dr_pos_err, 'r-')
axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Error (m)')
axes[1].set_title('DR Position Drift Over Time'); axes[1].grid(True, alpha=0.3)

axes[2].plot(t, np.degrees(yaw_gt), 'k-', label='GT')
axes[2].plot(t, np.degrees(psi_dr), 'r--', label='DR')
axes[2].set_xlabel('Time (s)'); axes[2].set_ylabel('Yaw (deg)')
axes[2].set_title('Heading: GT vs DR'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Compute the drift error (what the PINN will learn)

The correction target is simply: `δ = ground_truth − dead_reckoning`.  
This is a smooth, slowly-growing signal — **much** easier for a network to learn  
than absolute state from scratch.

In [ ]:
# Drift error = GT - DR  (what the PINN must learn to predict)
dpx  = (px_gt  - px_dr).astype(np.float32)
dpy  = (py_gt  - py_dr).astype(np.float32)
dvx  = (vx_gt  - vx_dr).astype(np.float32)
dvy  = (vy_gt  - vy_dr).astype(np.float32)
dpsi = (yaw_gt - psi_dr).astype(np.float32)

print(f"Correction signal stats:")
for name, arr in [("δpx", dpx), ("δpy", dpy), ("δvx", dvx), ("δvy", dvy), ("δψ", dpsi)]:
    print(f"  {name}: mean={arr.mean():.4f}  std={arr.std():.4f}  range=[{arr.min():.3f}, {arr.max():.3f}]")

## 5. Normalize & build tensors

In [ ]:
t_f32 = t.astype(np.float32)

# ── Normalize time input ──
t_mean, t_std = t_f32.mean(), t_f32.std()
t_norm = (t_f32 - t_mean) / t_std

# ── Normalize IMU inputs ──
ax_f32 = ax.astype(np.float32)
ay_f32 = ay.astype(np.float32)
wz_f32 = wz.astype(np.float32)

ax_mean, ax_std = ax_f32.mean(), ax_f32.std()
ay_mean, ay_std = ay_f32.mean(), ay_f32.std()
wz_mean, wz_std = wz_f32.mean(), wz_f32.std()

ax_norm = (ax_f32 - ax_mean) / ax_std
ay_norm = (ay_f32 - ay_mean) / ay_std
wz_norm = (wz_f32 - wz_mean) / wz_std

# ── Normalize correction targets ──
dpx_mean, dpx_std = dpx.mean(), max(dpx.std(), 1e-6)
dpy_mean, dpy_std = dpy.mean(), max(dpy.std(), 1e-6)
dvx_mean, dvx_std = dvx.mean(), max(dvx.std(), 1e-6)
dvy_mean, dvy_std = dvy.mean(), max(dvy.std(), 1e-6)
dpsi_mean, dpsi_std = dpsi.mean(), max(dpsi.std(), 1e-6)

dpx_norm  = (dpx  - dpx_mean)  / dpx_std
dpy_norm  = (dpy  - dpy_mean)  / dpy_std
dvx_norm  = (dvx  - dvx_mean)  / dvx_std
dvy_norm  = (dvy  - dvy_mean)  / dvy_std
dpsi_norm = (dpsi - dpsi_mean) / dpsi_std

print(f"Correction targets normalized. All should have std≈1:")
for name, arr in [("δpx", dpx_norm), ("δpy", dpy_norm), ("δvx", dvx_norm),
                   ("δvy", dvy_norm), ("δψ", dpsi_norm)]:
    print(f"  {name}: mean={arr.mean():.4f}  std={arr.std():.4f}")

In [ ]:
# Build tensors
# Input: normalized (t, ax, ay, wz)
X = torch.tensor(np.stack([t_norm, ax_norm, ay_norm, wz_norm], axis=1), dtype=torch.float32)
# Target: normalized corrections
Y = torch.tensor(np.stack([dpx_norm, dpy_norm, dvx_norm, dvy_norm, dpsi_norm], axis=1), dtype=torch.float32)
# Time with grad for physics
T = torch.tensor(t_norm, dtype=torch.float32).unsqueeze(1).requires_grad_(True)

# Dead-reckoning states as tensors (for adding correction in physics loss)
DR = torch.tensor(np.stack([
    px_dr.astype(np.float32), py_dr.astype(np.float32),
    vx_dr.astype(np.float32), vy_dr.astype(np.float32),
    psi_dr.astype(np.float32)
], axis=1), dtype=torch.float32)

# ── Split (70/15/15 by time) ──
n_train, n_val = int(0.7 * N), int(0.15 * N)

X_train, Y_train, T_train, DR_train = X[:n_train], Y[:n_train], T[:n_train], DR[:n_train]
X_val,   Y_val,   T_val,   DR_val   = X[n_train:n_train+n_val], Y[n_train:n_train+n_val], T[n_train:n_train+n_val], DR[n_train:n_train+n_val]
X_test,  Y_test,  T_test,  DR_test  = X[n_train+n_val:], Y[n_train+n_val:], T[n_train+n_val:], DR[n_train+n_val:]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

## 6. Model & physics loss

The network predicts **normalized corrections** `δ_norm`.  
The physics loss enforces that `(DR + denorm(δ))` satisfies the 2D INS equations:  
- `ṗx = vx`, `ṗy = vy`  
- `v̇x = ax·cos(ψ) − ay·sin(ψ)`, `v̇y = ax·sin(ψ) + ay·cos(ψ)`  
- `ψ̇ = wz`

In [ ]:
class CorrectionPINN(torch.nn.Module):
    """Predicts normalized drift corrections δ(t, imu)."""
    def __init__(self, hidden=128, n_layers=5):
        super().__init__()
        layers = []
        dims = [4] + [hidden] * n_layers + [5]
        for i in range(len(dims) - 1):
            layers.append(torch.nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:
                layers.append(torch.nn.Tanh())
        self.net = torch.nn.Sequential(*layers)

    def forward(self, x, t):
        inp = torch.cat([t, x[:, 1:4]], dim=1)
        return self.net(inp)

In [ ]:
def physics_loss(delta_norm, inputs_t, imu_norm, dr_states, np_dict):
    """
    Enforce 2D INS on the CORRECTED state = DR + denorm(δ).

    delta_norm: (N,5) normalized corrections from network
    inputs_t:   (N,1) normalized time (requires grad)
    imu_norm:   (N,3) normalized [ax, ay, wz]
    dr_states:  (N,5) dead-reckoning [px, py, vx, vy, psi] in real units
    np_dict:    normalization params on device
    """
    # ── Denormalize corrections to real units ──
    d_px  = delta_norm[:, 0:1] * np_dict["dpx_std"]  + np_dict["dpx_mean"]
    d_py  = delta_norm[:, 1:2] * np_dict["dpy_std"]  + np_dict["dpy_mean"]
    d_vx  = delta_norm[:, 2:3] * np_dict["dvx_std"]  + np_dict["dvx_mean"]
    d_vy  = delta_norm[:, 3:4] * np_dict["dvy_std"]  + np_dict["dvy_mean"]
    d_psi = delta_norm[:, 4:5] * np_dict["dpsi_std"] + np_dict["dpsi_mean"]

    # ── Corrected state = DR + correction ──
    px  = dr_states[:, 0:1] + d_px
    py  = dr_states[:, 1:2] + d_py
    vx  = dr_states[:, 2:3] + d_vx
    vy  = dr_states[:, 3:4] + d_vy
    psi = dr_states[:, 4:5] + d_psi

    # ── Time derivatives of corrected state ──
    # Only the correction term needs autograd (DR is a constant w.r.t. network params)
    ones = torch.ones_like(px)
    dpx_dt  = torch.autograd.grad(px,  inputs_t, ones, create_graph=True)[0] / np_dict["t_std"]
    dpy_dt  = torch.autograd.grad(py,  inputs_t, ones, create_graph=True)[0] / np_dict["t_std"]
    dvx_dt  = torch.autograd.grad(vx,  inputs_t, ones, create_graph=True)[0] / np_dict["t_std"]
    dvy_dt  = torch.autograd.grad(vy,  inputs_t, ones, create_graph=True)[0] / np_dict["t_std"]
    dpsi_dt = torch.autograd.grad(psi, inputs_t, ones, create_graph=True)[0] / np_dict["t_std"]

    # ── Denormalize IMU ──
    ax_real = imu_norm[:, 0:1] * np_dict["ax_std"] + np_dict["ax_mean"]
    ay_real = imu_norm[:, 1:2] * np_dict["ay_std"] + np_dict["ay_mean"]
    wz_real = imu_norm[:, 2:3] * np_dict["wz_std"] + np_dict["wz_mean"]

    # ── Physics residuals ──
    r1 = dpx_dt - vx
    r2 = dpy_dt - vy
    r3 = dvx_dt - (ax_real * torch.cos(psi) - ay_real * torch.sin(psi))
    r4 = dvy_dt - (ax_real * torch.sin(psi) + ay_real * torch.cos(psi))
    r5 = dpsi_dt - wz_real

    # ── Balanced residuals ──
    vel_scale = max(float(np_dict["dvx_std"]), float(np_dict["dvy_std"]), 0.1)
    acc_scale = max(float(np_dict["ax_std"]), float(np_dict["ay_std"]), 0.1)
    yaw_scale = max(float(np_dict["dpsi_std"]), 0.1)

    loss = (
        (r1 / vel_scale).pow(2).mean() +
        (r2 / vel_scale).pow(2).mean() +
        (r3 / acc_scale).pow(2).mean() +
        (r4 / acc_scale).pow(2).mean() +
        (r5 / yaw_scale).pow(2).mean()
    ) / 5.0

    return loss

## 7. Training loop

In [ ]:
def train_correction_pinn(model, X_tr, Y_tr, T_tr, DR_tr,
                           epochs=10000, lr=1e-3, lam=0.5,
                           lam_warmup=2000, np_dict=None):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    X_tr  = X_tr.to(device)
    Y_tr  = Y_tr.to(device)
    T_tr  = T_tr.to(device).requires_grad_(True)
    DR_tr = DR_tr.to(device)

    history = {"epoch": [], "data_loss": [], "phys_loss": [], "total_loss": []}

    for ep in range(epochs):
        optimizer.zero_grad()

        delta_pred = model(X_tr, T_tr)
        imu_for_phys = X_tr[:, 1:4]

        L_data = torch.nn.functional.mse_loss(delta_pred, Y_tr)
        L_phys = physics_loss(delta_pred, T_tr, imu_for_phys, DR_tr, np_dict)

        lam_cur = lam * min(1.0, ep / max(lam_warmup, 1))
        L_total = L_data + lam_cur * L_phys

        L_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        if ep % 500 == 0:
            print(f"Ep {ep:5d} | Data: {L_data:.6f} | "
                  f"Phys: {L_phys:.6f} | λ: {lam_cur:.4f} | Total: {L_total:.6f}")

        history["epoch"].append(ep)
        history["data_loss"].append(L_data.item())
        history["phys_loss"].append(L_phys.item())
        history["total_loss"].append(L_total.item())

    return model, history

In [ ]:
# ── Pack normalization params ──
def to_t(x):
    return torch.tensor(float(x), dtype=torch.float32, device=device)

np_dict = {
    "t_std":     to_t(t_std),
    "ax_mean":   to_t(ax_mean),   "ax_std":   to_t(ax_std),
    "ay_mean":   to_t(ay_mean),   "ay_std":   to_t(ay_std),
    "wz_mean":   to_t(wz_mean),   "wz_std":   to_t(wz_std),
    "dpx_mean":  to_t(dpx_mean),  "dpx_std":  to_t(dpx_std),
    "dpy_mean":  to_t(dpy_mean),  "dpy_std":  to_t(dpy_std),
    "dvx_mean":  to_t(dvx_mean),  "dvx_std":  to_t(dvx_std),
    "dvy_mean":  to_t(dvy_mean),  "dvy_std":  to_t(dvy_std),
    "dpsi_mean": to_t(dpsi_mean), "dpsi_std": to_t(dpsi_std),
}

In [ ]:
model = CorrectionPINN(hidden=128, n_layers=5)
model, history = train_correction_pinn(
    model, X_train, Y_train, T_train, DR_train,
    epochs=10000, lr=1e-3, lam=0.5, lam_warmup=2000,
    np_dict=np_dict
)

## 8. Evaluate

In [ ]:
# Correction denormalization stats
corr_means = np.array([dpx_mean, dpy_mean, dvx_mean, dvy_mean, dpsi_mean])
corr_stds  = np.array([dpx_std,  dpy_std,  dvx_std,  dvy_std,  dpsi_std])

# Ground truth for test set
Y_raw = np.stack([px_gt, py_gt, vx_gt, vy_gt, yaw_gt], axis=1).astype(np.float32)
Y_test_raw = Y_raw[n_train + n_val:]

# DR for test set
DR_test_np = np.stack([px_dr, py_dr, vx_dr, vy_dr, psi_dr], axis=1).astype(np.float32)
DR_test_np = DR_test_np[n_train + n_val:]


def evaluate(model, X_te, T_te, DR_test_np, Y_te_raw, corr_means, corr_stds):
    model.eval()
    with torch.no_grad():
        delta_norm = model(X_te.to(device), T_te.to(device)).cpu().numpy()

    # Denormalize corrections
    delta_real = delta_norm * corr_stds + corr_means

    # Corrected = DR + correction
    pred = DR_test_np + delta_real
    gt   = Y_te_raw

    pos_err  = np.sqrt((pred[:,0]-gt[:,0])**2 + (pred[:,1]-gt[:,1])**2)
    vel_err  = np.sqrt((pred[:,2]-gt[:,2])**2 + (pred[:,3]-gt[:,3])**2)
    head_err = np.abs(np.arctan2(np.sin(pred[:,4]-gt[:,4]), np.cos(pred[:,4]-gt[:,4])))

    return pos_err, vel_err, head_err, pred


pe, ve, he, pred_test = evaluate(model, X_test, T_test, DR_test_np, Y_test_raw, corr_means, corr_stds)

# Also get DR-only errors for comparison
dr_pe = np.sqrt((DR_test_np[:,0]-Y_test_raw[:,0])**2 + (DR_test_np[:,1]-Y_test_raw[:,1])**2)
dr_ve = np.sqrt((DR_test_np[:,2]-Y_test_raw[:,2])**2 + (DR_test_np[:,3]-Y_test_raw[:,3])**2)
dr_he = np.abs(np.arctan2(np.sin(DR_test_np[:,4]-Y_test_raw[:,4]),
                           np.cos(DR_test_np[:,4]-Y_test_raw[:,4])))

In [ ]:
print(f"{'Model':<20} {'Pos RMSE (m)':<15} {'Vel RMSE (m/s)':<16} {'Head MAE (°)':<14}")
print("-" * 65)
print(f"{'DR only':<20} {np.sqrt((dr_pe**2).mean()):<15.4f} "
      f"{np.sqrt((dr_ve**2).mean()):<16.4f} {np.degrees(dr_he.mean()):<14.4f}")
print(f"{'DR + PINN corr':<20} {np.sqrt((pe**2).mean()):<15.4f} "
      f"{np.sqrt((ve**2).mean()):<16.4f} {np.degrees(he.mean()):<14.4f}")

In [ ]:
def smooth(arr, window=50):
    return pd.Series(arr).rolling(window, center=True).mean().values

t_test_sec = t[n_train+n_val:n_train+n_val+len(pe)].astype(np.float32)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── Position error over time ──
axes[0,0].plot(t_test_sec, smooth(dr_pe), label='DR only', color='red', alpha=0.7)
axes[0,0].plot(t_test_sec, smooth(pe), label='DR + PINN', color='#2563eb')
axes[0,0].set_title('Position Error'); axes[0,0].set_xlabel('Time (s)')
axes[0,0].set_ylabel('Error (m)'); axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

# ── Velocity error over time ──
axes[0,1].plot(t_test_sec, smooth(dr_ve), label='DR only', color='red', alpha=0.7)
axes[0,1].plot(t_test_sec, smooth(ve), label='DR + PINN', color='#2563eb')
axes[0,1].set_title('Velocity Error'); axes[0,1].set_xlabel('Time (s)')
axes[0,1].set_ylabel('Error (m/s)'); axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

# ── 2D trajectory ──
axes[1,0].plot(Y_test_raw[:,0], Y_test_raw[:,1], 'k-', label='Ground Truth', linewidth=2)
axes[1,0].plot(DR_test_np[:,0], DR_test_np[:,1], 'r--', label='DR only', linewidth=1)
axes[1,0].plot(pred_test[:,0], pred_test[:,1], 'b-', label='DR + PINN', linewidth=1.5)
axes[1,0].set_title('2D Trajectory (Test Set)'); axes[1,0].set_xlabel('X (m)')
axes[1,0].set_ylabel('Y (m)'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)
axes[1,0].set_aspect('equal')

# ── Training curves ──
axes[1,1].semilogy(history['epoch'], history['data_loss'], label='Data loss')
axes[1,1].semilogy(history['epoch'], history['phys_loss'], label='Physics loss')
axes[1,1].semilogy(history['epoch'], history['total_loss'], label='Total loss', color='black', alpha=0.5)
axes[1,1].set_title('Training Curves'); axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Loss (log)'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('v3_results.png', dpi=150, bbox_inches='tight')
plt.show()